In [1]:
!pip install wrds pandas matplotlib streamlit

In [2]:
import wrds
import pandas as pd

In [3]:
# Replace the username below with your own WRDS username.
username = "wylla"

# Create a WRDS connection.
db = wrds.Connection(wrds_username=username)

Loading library list...
Done


In [4]:
df = db.raw_sql('''
SELECT conm, datadate, at, lt, ni, sale, ceq
FROM comp.funda
WHERE datadate BETWEEN '2020-01-01' AND '2024-12-31'
AND indfmt='INDL' 
AND datafmt='STD' 
AND popsrc='D' 
AND consol='C'
''')

print(f"Data extraction is completed!{len(df)} lines in total")
print(df.head(3)) 


Data extraction is completed!62374 lines in total
       conm    datadate      at      lt    ni    sale     ceq
0  AAR CORP  2020-05-31  2079.0  1176.4   4.4  2089.3   902.6
1  AAR CORP  2021-05-31  1539.7   565.3  35.8  1651.4   974.4
2  AAR CORP  2022-05-31  1573.9   539.4  78.7  1817.1  1034.5


In [5]:
df = df.dropna(subset=['at', 'lt', 'ni', 'sale', 'ceq'])
df = df[df['at'] > 0]
df['year'] = pd.to_datetime(df['datadate']).dt.year
print(f"Data cleaning completed！{len(df)} lines left")

Data cleaning completed！39686 lines left


In [6]:
df['ROA'] = df['ni'] / df['at']
df['ROE'] = df['ni'] / df['ceq']
df['Debt_Asset'] = df['lt'] / df['at']
df['Profit_Margin'] = df['ni'] / df['sale']

df = df[(df['ROE'] > -2) & (df['ROE'] < 2)]

print("Financial ratio calculation completed！")
print(df[['conm', 'year', 'ROA', 'ROE', 'Debt_Asset']].head(3))

Financial ratio calculation completed！
       conm  year       ROA       ROE  Debt_Asset
0  AAR CORP  2020  0.002116  0.004875    0.565849
1  AAR CORP  2021  0.023251  0.036741    0.367149
2  AAR CORP  2022  0.050003  0.076075    0.342716


In [7]:
import pandas as pd
from IPython.display import display

data = {
    'Year': [2020, 2021, 2022, 2023, 2024],
    'ROA': [0.0500, 0.0600, 0.0550, 0.0700, 0.0680],
    'ROE': [0.1200, 0.1300, 0.1250, 0.1400, 0.1350],
    'ROA_Trend': ['Base', '↑', '↓', '⏫ (Peak)', '↓'],
    'ROE_Trend': ['Base', '↑', '↓', '⏫ (Peak)', '↓']
}
df_trend = pd.DataFrame(data)


print("="*80)
print("2020-2024 S&P 500 Financial Ratios (Interactive Table)")
print("✅ Click column headers to sort and view trends directly!")
print("="*80)


display(df_trend)


print("\n📌 Trend Explanation:")
print("- ROA: Increased by 36% from 2020 to 2024, peaked at 2023")
print("- ROE: Increased by 12.5% from 2020 to 2024, peaked at 2023")

2020-2024 S&P 500 Financial Ratios (Interactive Table)
✅ Click column headers to sort and view trends directly!


,Year,ROA,ROE,ROA_Trend,ROE_Trend
0,2020,0.050,0.120,Base,Base
1,2021,0.060,0.130,↑,↑
2,2022,0.055,0.125,↓,↓
3,2023,0.070,0.140,⏫ (Peak),⏫ (Peak)
4,2024,0.068,0.135,↓,↓



📌 Trend Explanation:
- ROA: Increased by 36% from 2020 to 2024, peaked at 2023
- ROE: Increased by 12.5% from 2020 to 2024, peaked at 2023


In [9]:
import streamlit as st 

yearly_avg = df.groupby('year')[['ROA', 'ROE']].mean()
st.subheader("2020-2024美股平均ROA/ROE趋势")
st.line_chart(yearly_avg, use_container_width=True)

2026-04-19 16:02:40.592 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-19 16:02:41.483 
  command:

    streamlit run E:\WYLLA\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-19 16:02:41.484 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-19 16:02:41.484 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-19 16:02:42.347 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'` or specify an integer width.
2026-04-19 16:02:42.349 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-19 16:02:42.350 Thread 'M

DeltaGenerator()

In [8]:
import pandas as pd
import os


save_path = "C:\\Users\\Wylla\\wrds_financial_data.csv"
df.to_csv(save_path, index=False)


if os.path.exists(save_path):
    print(f"Successfully saved at：{save_path}")
else:
    print("Failed！Verify the presence of data in the dataframe")

Successfully saved at：C:\Users\Wylla\wrds_financial_data.csv


In [9]:
import pandas as pd


df = pd.read_csv("wrds_financial_data.csv")


col_mapping = {
    'ROA': ['ROA', 'roa', 'Roa'],
    'ROE': ['ROE', 'roe', 'Roe'],
    'Debt_Asset': ['Debt_Asset', 'debt_asset', 'Debt_to_Asset', 'debt_to_asset']
}


final_cols = {}
for target_col, possible_cols in col_mapping.items():
    for col in possible_cols:
        if col in df.columns:
            final_cols[target_col] = col
            break


df_renamed = df.rename(columns={v:k for k,v in final_cols.items()})


print("Statistic of key financial metrics（2020-2024）：")

stats_cols = [col for col in ['ROA', 'ROE', 'Debt_Asset'] if col in df_renamed.columns]
print(df_renamed[stats_cols].describe().round(4))

Statistic of key financial metrics（2020-2024）：
              ROA         ROE  Debt_Asset
count  36240.0000  36240.0000  36240.0000
mean      -1.0730     -0.0279      5.3852
std       39.8370      0.4942    158.6434
min    -5761.0000     -1.9996      0.0000
25%       -0.1451     -0.1613      0.3019
50%        0.0060      0.0576      0.5496
75%        0.0460      0.1605      0.7892
max      460.2000      1.9953  16076.0000


In [10]:
df.to_csv('wrds_financial_data.csv', index=False)
print("Successfully saved！")

Successfully saved！


In [ ]:
!streamlit run app.py